In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [2]:
from datetime import datetime

import numpy as np
import pandas as pd
from scipy.stats import norm

from src.analytics import VarianceCalculator
from src.core.schemas import OptionChain

## Test de _interpolate_constant_maturity

In [4]:
print(VarianceCalculator._interpolate_constant_maturity(0.04, 23, 0.0625, 37))

0.053875


## Test de _integrate_variance

In [5]:
s0 = 4000.0
r = 0.05
sigma = 0.20
days = 30
t = days / 365
f = s0 * np.exp(r * t)

In [6]:
k = np.arange(2000, 8001, 5)
d1 = (np.log(s0 / k) + (r + sigma**2 / 2) * t) / (sigma * np.sqrt(t))
d2 = d1 - sigma * np.sqrt(t)
call = s0 * norm.cdf(d1) - k * np.exp(-r * t) * norm.cdf(d2)
put = k * np.exp(-r * t) * norm.cdf(-d2) - s0 * norm.cdf(-d1)
price = np.where(k < f, put, call)

In [7]:
otm = pd.DataFrame({"strike": k, "option_type": np.where(k < f, "P", "C"), "bid": price, "ask": price, "mid_price": price})
chain = OptionChain(
    date=datetime(2026, 11, 1), maturity_days=days, underlying_price=s0, risk_free_rate=r, forward_price=f, options_data=otm
)

In [8]:
print(VarianceCalculator._integrate_variance(otm, chain, is_svix=False))  # attendu ≈ 0.0400
print(VarianceCalculator._integrate_variance(otm, chain, is_svix=True))  # attendu ≈ 0.0401

0.04000052204544159
0.04006634300452335
